# DAR analysis for mouse AD project

In [2]:
suppressPackageStartupMessages({
    library(Seurat)
    library(SeuratWrappers)
    library(patchwork)
    library(ggplot2)
    library(repr)
    library(gridExtra)
    library(edgeR)
    library(SingleCellExperiment)
    library(Matrix)
    library(scran)
    library(tidyverse)
    library(ggrepel)
    library(scater)
})
options(future.globals.maxSize = 1e9)
options(Seurat.object.assay.version = "v5")
options(ggrepel.max.overlaps = Inf)

## Load data

In [4]:
data.dir <- '/mouseAD/processed_data/'
meta.path <- file.path(data.dir, 'meta.data.txt')
meta.data <- read.csv(meta.path, header = TRUE, row.names = 1, sep = '\t')

In [5]:
meta.data.mouse <- meta.data |>
    dplyr::select(library_id, mouse_id, genotype, age, sex, intervention) |>
    dplyr::distinct(library_id, .keep_all = TRUE)

In [6]:
meta.data.mouse$intervention[meta.data.mouse$intervention == "na"] <- "Sed"
meta.data.mouse$genotype[meta.data.mouse$genotype == "5XFAD"] <- "AD"

In [8]:
all.paths <- list.files("Celltype_ATAC_Counts/")
for (path in all.paths) {
    celltype = strsplit(path, "*_each_mouse_atac_counts_celltype_peaks.tsv")[[1]][1]
    if (celltype %in% c("SMC-Peri", "RGL")) {
        next
    }
    print(celltype)
    flush.console()
    
    if (file.exists(file.path("DAR_fit_genotype_age_models/", paste0(celltype, ".model")))) {
        next
    }
    
    curr.datapath <- paste0("Celltype_ATAC_Counts/", path)
    curr.counts <- read.csv(
        curr.datapath,
        sep = "\t",
        header = TRUE,
        row.names = 1,
        check.names = FALSE
    )
    curr.meta <- meta.data.mouse[colnames(curr.counts), ]
    
    Genotype <- curr.meta$genotype
    Age <- curr.meta$age
    Intervention <- curr.meta$intervention
    Sex <- curr.meta$sex
    
    # No Intervention model
    Group <- paste(Genotype, Age, Sex, sep = ".")
    Group.levels <- c(
        'WT.3M.M', 'WT.3M.F',
        'AD.3M.M', 'AD.3M.F',
        'WT.9M.M', 'WT.9M.F',
        'AD.9M.M', 'AD.9M.F',
        'WT.18M.M', 'WT.18M.F',
        'AD.18M.M', 'AD.18M.F'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_intervention_models/', paste(celltype, 'model', sep = '.')))
    
    # No Sex model
    Group <- paste(Genotype, Age, Intervention, sep = ".")
    Group.levels <- c(
        'WT.3M.Sed', 'AD.3M.Sed',
        'WT.9M.Sed', 'WT.9M.Ex',
        'AD.9M.Sed', 'AD.9M.Ex',
        'WT.18M.Sed', 'WT.18M.Ex',
        'AD.18M.Sed', 'AD.18M.Ex'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_sex_models/', paste(celltype, 'model', sep = '.')))
    
    # Genotype x Age model
    Group <- paste(Genotype, Age, sep = ".")
    Group.levels <- c('WT.3M', 'AD.3M', 'WT.9M', 'AD.9M', 'WT.18M', 'AD.18M')
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_genotype_age_models/', paste(celltype, 'model', sep = '.')))
}

[1] "Astro"
[1] "CA1"
[1] "CA2"
[1] "CA3"
[1] "Cajal-Retzius"


Removing 2 rows with all zero counts

Removing 2 rows with all zero counts

Removing 2 rows with all zero counts



[1] "Chandelier"
[1] "Choroid-plexus"
[1] "DG"
[1] "Endo"


Removing 3 rows with all zero counts

Removing 3 rows with all zero counts

Removing 3 rows with all zero counts



[1] "IOL"
[1] "Lamp5"
[1] "Meis2"
[1] "Microglia"
[1] "NB"
[1] "Oligo"
[1] "OPC"
[1] "Pvalb"
[1] "PVM"


Removing 22 rows with all zero counts

Removing 22 rows with all zero counts

Removing 22 rows with all zero counts



[1] "Sncg"
[1] "Sst"
[1] "SUB_1"
[1] "SUB_2"
[1] "SUB_3"
[1] "SUB-ProS"
[1] "Vip"
[1] "VLMC"


Removing 13 rows with all zero counts

Removing 13 rows with all zero counts

Removing 13 rows with all zero counts



In [9]:
all.paths <- list.files("Celltype_ATAC_Counts/")
for (path in all.paths) {
    celltype = strsplit(path, "*_each_mouse_atac_counts_celltype_peaks.tsv")[[1]][1]
    if (celltype != "SMC-Peri") {
        next
    }
    print(celltype)
    flush.console()
    
    curr.datapath <- paste0("Celltype_ATAC_Counts/", path)
    curr.counts <- read.csv(
        curr.datapath,
        sep = "\t",
        header = TRUE,
        row.names = 1,
        check.names = FALSE
    )
    curr.meta <- meta.data.mouse[colnames(curr.counts), ]
    
    Genotype <- curr.meta$genotype
    Age <- curr.meta$age
    Intervention <- curr.meta$intervention
    Sex <- curr.meta$sex
    
    # No Intervention model
    Group <- paste(Genotype, Age, Sex, sep = ".")
    Group.levels <- c(
        'WT.3M.F',
        'AD.3M.M', 'AD.3M.F',
        'WT.9M.M', 'WT.9M.F',
        'AD.9M.M', 'AD.9M.F',
        'WT.18M.M', 'WT.18M.F',
        'AD.18M.M', 'AD.18M.F'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_intervention_models/', paste(celltype, 'model', sep = '.')))
    
    # No Sex model
    Group <- paste(Genotype, Age, Intervention, sep = ".")
    Group.levels <- c(
        'WT.3M.Sed', 'AD.3M.Sed',
        'WT.9M.Sed', 'WT.9M.Ex',
        'AD.9M.Sed', 'AD.9M.Ex',
        'WT.18M.Sed', 'WT.18M.Ex',
        'AD.18M.Sed', 'AD.18M.Ex'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_sex_models/', paste(celltype, 'model', sep = '.')))
    
    # Genotype x Age model
    Group <- paste(Genotype, Age, sep = ".")
    Group.levels <- c('WT.3M', 'AD.3M', 'WT.9M', 'AD.9M', 'WT.18M', 'AD.18M')
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_genotype_age_models/', paste(celltype, 'model', sep = '.')))
}

[1] "SMC-Peri"


Removing 40 rows with all zero counts

Removing 40 rows with all zero counts

Removing 40 rows with all zero counts



In [10]:
all.paths <- list.files("Celltype_ATAC_Counts/")
for (path in all.paths) {
    celltype = strsplit(path, "*_each_mouse_atac_counts_celltype_peaks.tsv")[[1]][1]
    if (celltype != "RGL") {
        next
    }
    print(celltype)
    flush.console()
    
    curr.datapath <- paste0("Celltype_ATAC_Counts/", path)
    curr.counts <- read.csv(
        curr.datapath,
        sep = "\t",
        header = TRUE,
        row.names = 1,
        check.names = FALSE
    )
    curr.meta <- meta.data.mouse[colnames(curr.counts), ]
    
    Genotype <- curr.meta$genotype
    Age <- curr.meta$age
    Intervention <- curr.meta$intervention
    Sex <- curr.meta$sex
    
    # No Intervention model
    Group <- paste(Genotype, Age, Sex, sep = ".")
    Group.levels <- c(
        'WT.3M.M', 'WT.3M.F',
        'AD.3M.M', 'AD.3M.F',
        'WT.9M.M', 'WT.9M.F',
        'AD.9M.M', 'AD.9M.F',
        'WT.18M.M', 'WT.18M.F',
        'AD.18M.M', 'AD.18M.F'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_intervention_models/', paste(celltype, 'model', sep = '.')))
    
    # No Sex model
    Group <- paste(Genotype, Age, Intervention, sep = ".")
    Group.levels <- c(
        'WT.3M.Sed', 'AD.3M.Sed',
        'WT.9M.Sed', 'WT.9M.Ex',
        'AD.9M.Sed', 'AD.9M.Ex',
        'WT.18M.Sed', 'WT.18M.Ex',
        'AD.18M.Sed', 'AD.18M.Ex'
    )
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_no_sex_models/', paste(celltype, 'model', sep = '.')))
    
    # Genotype x Age model
    Group <- paste(Genotype, Age, sep = ".")
    Group.levels <- c('WT.3M', 'AD.3M', 'WT.9M', 'AD.9M', 'WT.18M', 'AD.18M')
    Group <- factor(Group, levels = Group.levels)
    curr.dar <- DGEList(counts = curr.counts, group = Group, remove.zeros = TRUE)
    keep <- filterByExpr(curr.dar, min.count = 5, min.prop = 0.5)
    curr.dar <- curr.dar[keep, , keep.lib.sizes=FALSE]
    curr.dar <- calcNormFactors(curr.dar, method = 'TMM')
    curr.design <- model.matrix(~ 0 + Group)
    colnames(curr.design) <- levels(Group)
    curr.dar <- estimateDisp(curr.dar, curr.design, robust = TRUE)
    fit <- glmQLFit(curr.dar, curr.design)
    saveRDS(fit, file.path('DAR_fit_genotype_age_models/', paste(celltype, 'model', sep = '.')))
}

[1] "RGL"


Removing 17 rows with all zero counts

Removing 17 rows with all zero counts

Removing 17 rows with all zero counts

